In [ ]:
!git clone --depth 1 https://github.com/langchain-ai/lca-lc-foundations.git
%cd lca-lc-foundations
!pwd
!pip install -r requirements.txt



In [ ]:
from google.colab import userdata
from langchain.chat_models import init_chat_model
from tavily import TavilyClient




# Get the secret value from Colab's secrets manager
openai_key = userdata.get('OPENAI_API_KEY')



model = init_chat_model(model="gpt-5-nano", api_key=openai_key)

# Get the Tavily API key from Colab's secrets manager
tavily_key = userdata.get('TAVILY_API_KEY')


In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=tavily_key)

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent_chef = create_agent(
    model = model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent_chef.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
response = agent_chef.invoke(
    {"messages": [HumanMessage(content="I also have some indian spices, any changes you recommend?")]},
    config
)

print(response['messages'][-1].content)